In [1]:
import sqlite3
import pandas as pd
import pantab
from pathlib import Path

folder = Path(r"C:\Users\alrazz\Documents\Anonating files")


In [2]:
db_paths = list(folder.glob("*.db"))
#print (db_paths)

def load_texts_from_dbs(db_paths):
    dfs = {}

    for db_path in db_paths:
        with sqlite3.connect(db_path) as conn:
            dfs[db_path.stem] = pd.read_sql_query(
                'SELECT u, id, ts, text, TURKU_NLP, TURKU_NLP_sub, Tags, "web-register" FROM texts;',
                conn
            )

    return dfs

dfs = load_texts_from_dbs(db_paths)

#dfs["HI_clean_dedup"].head()

In [3]:
print(dfs["HI_clean_dedup"].columns)

Index(['u', 'id', 'ts', 'text', 'Turku_NLP', 'Turku_NLP_sub', 'Tags',
       'web-register'],
      dtype='object')


In [4]:
dfs_noNA = {
    name: df.dropna(subset=["Turku_NLP", "Turku_NLP_sub"])
    for name, df in dfs.items()
}

for name, df in dfs_noNA.items():
    print(name, df.shape)

ed (47, 8)
ed_2 (98, 8)
ed_3 (5, 8)
en (3, 8)
en_10 (17, 8)
en_11 (19, 8)
en_2 (3, 8)
en_3 (11, 8)
en_4 (8, 8)
en_5 (7, 8)
en_6 (16, 8)
en_7 (4, 8)
en_8 (4, 8)
en_9 (24, 8)
fi (147, 8)
HI_2_clean_dedup (85, 8)
HI_clean_dedup (61, 8)
HI_ID_LY_SP_clean2_dedup (6, 8)
ID_clean_dedup (225, 8)
it (6, 8)
it_2 (7, 8)
it_3 (91, 8)
lt (149, 8)
lt_2 (58, 8)
lt_3 (75, 8)
lt_4 (30, 8)
LY_clean_dedup (181, 8)
MT (148, 8)
nb (15, 8)
nb_2 (100, 8)
nb_3 (0, 8)
nb_4 (6, 8)
nb_5 (1, 8)
nb_6 (12, 8)
nb_7 (23, 8)
New_HI_ID_LY_OP_SP_clean_clean (89, 8)
ob (6, 8)
ob_2 (32, 8)
ob_3 (61, 8)
ob_4 (34, 8)
oi (1, 8)
oi_2 (80, 8)
oi_3 (35, 8)
oo (4, 8)
oo_2 (14, 8)
oo_3 (0, 8)
oo_4 (67, 8)
oo_5 (3, 8)
oo_6 (45, 8)
os (3, 8)
os_2 (106, 8)
Persian_data (1100, 8)
Persian_data3_clean (99, 8)
ra (0, 8)
ra_2 (10, 8)
ra_3 (44, 8)
ra_4 (76, 8)
re (92, 8)
re_2 (31, 8)
re_3 (36, 8)
re_4 (28, 8)
rs (151, 8)
rv (9, 8)
rv_2 (10, 8)
rv_3 (53, 8)
rv_4 (28, 8)
rv_5 (54, 8)
SP_clean_dedup (160, 8)
sr (97, 8)
sr_2 (95, 8)
sr_3 (27,

In [5]:
def process_turku(df, main_col="Turku_NLP", sub_col="Turku_NLP_sub"):
    """
    Processes a dataframe by splitting the given Turku columns,
    pairing elementwise, exploding, and fixing '-' entries.
    """

    df_copy = df.copy()

    # Step 1: Split into lists
    df_copy[f"{main_col}_split"] = df_copy[main_col].str.split(" ; ")
    df_copy[f"{sub_col}_split"] = df_copy[sub_col].str.split(" ; ")

    # Step 2: Pair elementwise
    df_copy["paired"] = df_copy.apply(
        lambda x: list(zip(x[f"{main_col}_split"], x[f"{sub_col}_split"])),
        axis=1
    )

    # Step 3: Explode
    exploded = df_copy.explode("paired").copy()

    # Step 4: Restore into two separate columns
    exploded[f"{main_col}_split"] = exploded["paired"].apply(
        lambda x: x[0] if pd.notna(x) else None
    )
    exploded[f"{sub_col}_split"] = exploded["paired"].apply(
        lambda x: x[1] if pd.notna(x) else None
    )

    exploded = exploded.drop(columns=["paired"])

    # Step 5: Replace "-" in sub column
    exploded[f"{sub_col}_split_modified"] = exploded[f"{sub_col}_split"]

    mask = (
        (exploded[f"{sub_col}_split"] == "-")
        & (exploded[f"{main_col}_split"] != "-")
    )

    exploded.loc[mask, f"{sub_col}_split_modified"] = exploded.loc[
        mask, f"{main_col}_split"
    ]

    return exploded

In [6]:
dfs_processed = {}

for name, df in dfs_noNA.items():
    if df.empty:
        continue

    processed = process_turku(df)
    processed["source"] = name
    dfs_processed[name] = processed

df_all = pd.concat(dfs_processed.values(), ignore_index=True)

In [7]:
df_all = pd.concat(dfs_processed.values(), ignore_index=True)

In [8]:
print(df_all["Turku_NLP_sub_split_modified"].unique())
print(df_all["Turku_NLP_split"].unique())

['ed' 'ID' 'on' 'ob' 'os' 'ra' 'rs' 'dtp' 'oo' 'ne' 'LY' 'nb' 'oe' 'oi'
 'lt' 'en' 'fi' 'av' 'it' 'oh' 'ds' 're' 'rv' 'MT' '-' 'sr']
['IP' 'ID' 'NA' 'OP' 'SP' 'IN' 'LY' 'HI' 'MT' '-']


In [9]:
print(len((df_all["Turku_NLP_sub_split_modified"].unique()))) #should be 26
print(len(df_all["Turku_NLP_split"].unique())) #should be 10

26
10


In [10]:
# Find erros like lt ;oh (where space after ; forgotten)
def find_value(dfs_processed, column, value):
    for name, df in dfs_processed.items():
        matches = df[df[column] == value]

        if not matches.empty:
            print(f"\n===== {name} ({len(matches)} matches) =====")
            display(matches)

In [11]:
find_value(
    dfs_processed,
    "Turku_NLP_sub_split_modified", #Turku_NLP_split
    "op" #for example lt ;oh
)

In [12]:
df_all.head()
#Turku_NLP_split is the main register
#Turku_NLP_sub_split is the sub register
#Turku_NLP_sub_split_modified for subregister "-" --> replaced by main register

,u,id,ts,text,Turku_NLP,Turku_NLP_sub,Tags,web-register,Turku_NLP_split,Turku_NLP_sub_split,Turku_NLP_sub_split_modified,source
0,http://roshangari.info/?p=35434,110ae9e775485f536c4f1cdea087cbd7,2020-01-25T19:50:48Z,e.shafagh@yahoo.com\n“تاریخ گواهی خواهد داد که...,IP ; ID,ed ; -,OTHLI,"{""MT"": 0.074, ""LY"": 0.084, ""SP"": 0.105, ""ID"": ...",IP,ed,ed,ed
1,http://roshangari.info/?p=35434,110ae9e775485f536c4f1cdea087cbd7,2020-01-25T19:50:48Z,e.shafagh@yahoo.com\n“تاریخ گواهی خواهد داد که...,IP ; ID,ed ; -,OTHLI,"{""MT"": 0.074, ""LY"": 0.084, ""SP"": 0.105, ""ID"": ...",ID,-,ID,ed
2,https://www.tinn.ir/%D8%A8%D8%AE%D8%B4-%D9%88%...,32962503dcdb08ee1bc4d2e9d5240bc7,2020-01-19T18:40:03Z,در کشورهای پیشرفته که حملونقل بر اساس پژوهشهای...,IP,ed,None,"{""MT"": 0.089, ""LY"": 0.085, ""SP"": 0.125, ""ID"": ...",IP,ed,ed,ed
3,http://p313.ir/post-124289.html,0e6562306fcb8db55e019445407cb487,2016-10-26T00:51:16Z,مهدی محمدی طی یادداشتی در روزنامه وطن امروز نو...,IP,ed,None,"{""MT"": 0.093, ""LY"": 0.063, ""SP"": 0.11900000000...",IP,ed,ed,ed
4,http://shakhesnews.com/%db%b6-%d8%af%d8%a7%d9%...,588d3acafdc57f5122849f3163c1d357,2016-10-27T18:45:54Z,شاخص : اگر مسیر مذاکرات در همین جهت ادامه یابد...,IP,ed,OTHLI,"{""MT"": 0.092, ""LY"": 0.07100000000000001, ""SP"":...",IP,ed,ed,ed


In [13]:
df_all.nunique()

u                               4449
id                              4449
ts                              4446
text                            4449
Turku_NLP                        190
Turku_NLP_sub                    472
Tags                              10
web-register                    4403
Turku_NLP_split                   10
Turku_NLP_sub_split               23
Turku_NLP_sub_split_modified      26
source                            68
dtype: int64

# Check Inconsistent IDs

In [16]:
# Columns you want to compare
cols_to_check = ["Turku_NLP", "Turku_NLP_sub"]

# Group by ID
grouped = df_all.groupby("id")

inconsistent_ids = []

for doc_id, group in grouped:
    # For each of the columns, check if there is more than one unique non-null value
    inconsistent = False
    for col in cols_to_check:
        unique_vals = group[col].dropna().unique()
        if len(unique_vals) > 1:
            inconsistent = True
    if inconsistent:
        inconsistent_ids.append(doc_id)

print("Inconsistent IDs:", inconsistent_ids)
print(f"Total inconsistent IDs: {len(inconsistent_ids)}")

# Optional: print detailed info like Claude did
for doc_id in inconsistent_ids:
    print("\nID", doc_id, "has inconsistent values:")
    display(df_all[df_all["id"] == doc_id][["id", "Turku_NLP", "Turku_NLP_sub", "source"]])


Inconsistent IDs: ['1fd4f8a811448f194ef56711577542de', '9f9e2e542c8cf378c64297b9152df3c8', 'b39c8345321e79f9c2f6d6cb31adecab', 'bbb7ecdd45d8a5457b4a924697bdf350', 'e03efa73087a28a5907375428a7f5f4b', 'ed488ebc1091395805fafbcfd8fd8c57', 'f3bdbd3c303bfbfcf28d3517f9242f41', 'fdb9a8b4ba723d2a652683996e335085', 'fed7746f5aaa046e7d0f5c4fae268306']
Total inconsistent IDs: 9

ID 1fd4f8a811448f194ef56711577542de has inconsistent values:


,id,Turku_NLP,Turku_NLP_sub,source
1986,1fd4f8a811448f194ef56711577542de,LY ; IN,- ; dtp,LY_clean_dedup
1987,1fd4f8a811448f194ef56711577542de,LY ; IN,- ; dtp,LY_clean_dedup
3230,1fd4f8a811448f194ef56711577542de,OP ; LY,oo ; -,oo_6
3231,1fd4f8a811448f194ef56711577542de,OP ; LY,oo ; -,oo_6



ID 9f9e2e542c8cf378c64297b9152df3c8 has inconsistent values:


,id,Turku_NLP,Turku_NLP_sub,source
3343,9f9e2e542c8cf378c64297b9152df3c8,SP,os,os_2
5916,9f9e2e542c8cf378c64297b9152df3c8,SP ; OP,os ; rs,SP_clean_dedup
5917,9f9e2e542c8cf378c64297b9152df3c8,SP ; OP,os ; rs,SP_clean_dedup



ID b39c8345321e79f9c2f6d6cb31adecab has inconsistent values:


...

,id,Turku_NLP,Turku_NLP_sub,source
2602,f3bdbd3c303bfbfcf28d3517f9242f41,IN ; HI,dtp ; oh,New_HI_ID_LY_OP_SP_clean_clean
2603,f3bdbd3c303bfbfcf28d3517f9242f41,IN ; HI,dtp ; oh,New_HI_ID_LY_OP_SP_clean_clean
2840,f3bdbd3c303bfbfcf28d3517f9242f41,IN,oi,oi_2



ID fdb9a8b4ba723d2a652683996e335085 has inconsistent values:


,id,Turku_NLP,Turku_NLP_sub,source
3348,fdb9a8b4ba723d2a652683996e335085,SP,os,os_2
5914,fdb9a8b4ba723d2a652683996e335085,SP ; OP,os ; rs,SP_clean_dedup
5915,fdb9a8b4ba723d2a652683996e335085,SP ; OP,os ; rs,SP_clean_dedup



ID fed7746f5aaa046e7d0f5c4fae268306 has inconsistent values:


,id,Turku_NLP,Turku_NLP_sub,source
1040,fed7746f5aaa046e7d0f5c4fae268306,HI,re,ID_clean_dedup
2371,fed7746f5aaa046e7d0f5c4fae268306,NA ; HI,nb ; re,nb_2
2372,fed7746f5aaa046e7d0f5c4fae268306,NA ; HI,nb ; re,nb_2


In [14]:
df_all.to_csv('Second_try_1.CSV', index=False)